In [0]:
CREATE OR REPLACE TABLE workspace.default.dim_empresa_gold AS
SELECT
    ROW_NUMBER() OVER (ORDER BY EMPRESA_NOME) AS EMPRESA_ID,
    EMPRESA_NOME,
    EMPRESA_NACIONALIDADE
FROM (
    SELECT DISTINCT
        EMPRESA_NOME,
        EMPRESA_NACIONALIDADE
    FROM workspace.default.dados_anac_silver
    WHERE EMPRESA_NOME IS NOT NULL
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.dim_empresa_gold;

EMPRESA_ID,EMPRESA_NOME,EMPRESA_NACIONALIDADE
1,"21 AIR, LLC",ESTRANGEIRA
2,"7 AIR, LLC.",ESTRANGEIRA
3,ABAETÉ LINHAS AÉREAS S.A.,BRASILEIRA
4,ABSA - AEROLINHAS BRASILEIRAS S.A.,BRASILEIRA
5,"ABX AIR, INC.",ESTRANGEIRA
6,ACG AIR CARGO GERMANY GMBH,ESTRANGEIRA
7,ACT AIRLINES,ESTRANGEIRA
8,AEROENLACES NACIONALES S.A.- VIVA AEROBUS,ESTRANGEIRA
9,AEROFLOT- AEROLÍNEAS INTERNACIONAIS DA RÚSSIA,ESTRANGEIRA
10,AEROLINEAS ARGENTINAS S/A,ESTRANGEIRA


In [0]:
CREATE OR REPLACE TABLE workspace.default.dim_aeroporto_gold AS
SELECT
    ROW_NUMBER() OVER (ORDER BY AEROPORTO_NOME) AS AEROPORTO_ID,
    AEROPORTO_NOME,
    PAIS
FROM (
    SELECT DISTINCT
        AEROPORTO_DE_ORIGEM_NOME AS AEROPORTO_NOME,
        AEROPORTO_DE_ORIGEM_PAIS AS PAIS
    FROM workspace.default.dados_anac_silver
    WHERE AEROPORTO_DE_ORIGEM_NOME IS NOT NULL

    UNION

    SELECT DISTINCT
        AEROPORTO_DE_DESTINO_NOME AS AEROPORTO_NOME,
        AEROPORTO_DE_DESTINO_PAIS AS PAIS
    FROM workspace.default.dados_anac_silver
    WHERE AEROPORTO_DE_DESTINO_NOME IS NOT NULL
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.dim_aeroporto_gold;

AEROPORTO_ID,AEROPORTO_NOME,PAIS
1,AB DHABI INTERNATIONAL,EMIRADOS ÁRABES UNIDOS
2,"ABBOTSFORD, BRITISH COLUMBIA",CANADÁ
3,ABIDJAN,COSTA DO MARFIM
4,"ABUJA, FCT",NIGÉRIA
5,ACAPULCO/GEN.JUAN ALVAREZ,MÉXICO
6,ACCRA,GANA
7,ADANA,TURQUIA
8,ADDIS ABABA,ETIÓPIA
9,"ADDISON, TEXAS",ESTADOS UNIDOS DA AMÉRICA
10,ADUSTINA,BRASIL


In [0]:
CREATE OR REPLACE TABLE workspace.default.dim_natureza_gold AS
SELECT
    ROW_NUMBER() OVER (ORDER BY NATUREZA) AS NATUREZA_ID,
    NATUREZA
FROM (
    SELECT DISTINCT NATUREZA
    FROM workspace.default.dados_anac_silver
    WHERE NATUREZA IS NOT NULL
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.dim_natureza_gold ORDER BY NATUREZA_ID;

NATUREZA_ID,NATUREZA
1,DOMÉSTICA
2,INTERNACIONAL


In [0]:
CREATE OR REPLACE TABLE workspace.default.dim_grupo_voo_gold AS
SELECT
    ROW_NUMBER() OVER (ORDER BY GRUPO_DE_VOO) AS GRUPO_VOO_ID,
    GRUPO_DE_VOO
FROM (
    SELECT DISTINCT GRUPO_DE_VOO
    FROM workspace.default.dados_anac_silver
    WHERE GRUPO_DE_VOO IS NOT NULL
);

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.dim_grupo_voo_gold ORDER BY GRUPO_VOO_ID;

GRUPO_VOO_ID,GRUPO_DE_VOO
1,IMPRODUTIVO
2,NÃO IDENTIFICADO
3,NÃO REGULAR
4,REGULAR


In [0]:
CREATE OR REPLACE TABLE workspace.default.fato_voos_gold AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY s.EMPRESA_NOME, s.ANO, s.MES,
                 s.AEROPORTO_DE_ORIGEM_NOME, s.AEROPORTO_DE_DESTINO_NOME,
                 s.NATUREZA, s.GRUPO_DE_VOO
    ) AS VOO_ID,

    e.EMPRESA_ID,
    ao.AEROPORTO_ID AS AEROPORTO_ORIGEM_ID,
    ad.AEROPORTO_ID AS AEROPORTO_DESTINO_ID,
    n.NATUREZA_ID,
    g.GRUPO_VOO_ID,

    s.ANO,
    s.MES,
    CONCAT(LPAD(CAST(s.MES AS STRING), 2, '0'), '/', CAST(s.ANO AS STRING)) AS MES_ANO,

    s.PASSAGEIROS_PAGOS,
    s.PASSAGEIROS_GRATIS,
    s.DISTANCIA_VOADA_KM,
    s.ASSENTOS,
    s.DECOLAGENS

FROM workspace.default.dados_anac_silver s

LEFT JOIN workspace.default.dim_empresa_gold e
    ON s.EMPRESA_NOME = e.EMPRESA_NOME

LEFT JOIN workspace.default.dim_aeroporto_gold ao
    ON s.AEROPORTO_DE_ORIGEM_NOME = ao.AEROPORTO_NOME

LEFT JOIN workspace.default.dim_aeroporto_gold ad
    ON s.AEROPORTO_DE_DESTINO_NOME = ad.AEROPORTO_NOME

LEFT JOIN workspace.default.dim_natureza_gold n
    ON s.NATUREZA = n.NATUREZA

LEFT JOIN workspace.default.dim_grupo_voo_gold g
    ON s.GRUPO_DE_VOO = g.GRUPO_DE_VOO;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.fato_voos_gold LIMIT 10;

VOO_ID,EMPRESA_ID,AEROPORTO_ORIGEM_ID,AEROPORTO_DESTINO_ID,NATUREZA_ID,GRUPO_VOO_ID,ANO,MES,MES_ANO,PASSAGEIROS_PAGOS,PASSAGEIROS_GRATIS,DISTANCIA_VOADA_KM,ASSENTOS,DECOLAGENS
1,1,556,136,2,3,2017,3,03/2017,0,0,null,null,null
2,1,556,562,2,3,2017,3,03/2017,0,0,1701,0,1
3,1,562,136,2,3,2017,3,03/2017,0,0,1787,0,1
4,1,598,136,2,3,2017,3,03/2017,0,0,null,null,null
5,1,598,556,2,3,2017,3,03/2017,0,0,2193,0,1
6,1,598,562,2,3,2017,3,03/2017,0,0,null,null,null
7,1,147,587,2,3,2017,9,09/2017,0,0,3895,0,1
8,1,147,598,2,3,2017,9,09/2017,0,0,null,null,null
9,1,587,598,2,3,2017,9,09/2017,0,0,2243,0,1
10,1,598,147,2,3,2017,9,09/2017,0,0,5808,0,1


In [0]:
-- Comentários no Unity Catalog para tabelas da camada Gold

COMMENT ON TABLE workspace.default.dim_empresa_gold IS 'Dimensão Gold: empresas aéreas da ANAC. 328 empresas com ID único. Fonte: dados_anac_silver.';
COMMENT ON COLUMN workspace.default.dim_empresa_gold.EMPRESA_ID IS 'Chave primária sequencial, ordenada por EMPRESA_NOME';
COMMENT ON COLUMN workspace.default.dim_empresa_gold.EMPRESA_NOME IS 'Nome completo da empresa aérea conforme ANAC';
COMMENT ON COLUMN workspace.default.dim_empresa_gold.EMPRESA_NACIONALIDADE IS 'Nacionalidade: BRASILEIRA ou ESTRANGEIRA';

COMMENT ON TABLE workspace.default.dim_aeroporto_gold IS 'Dimensão Gold: aeroportos da ANAC (origem e destino unificados). 1.058 aeroportos. Fonte: dados_anac_silver.';
COMMENT ON COLUMN workspace.default.dim_aeroporto_gold.AEROPORTO_ID IS 'Chave primária sequencial, ordenada por AEROPORTO_NOME';
COMMENT ON COLUMN workspace.default.dim_aeroporto_gold.AEROPORTO_NOME IS 'Nome do aeroporto conforme ANAC';
COMMENT ON COLUMN workspace.default.dim_aeroporto_gold.PAIS IS 'País onde o aeroporto está localizado';

COMMENT ON TABLE workspace.default.dim_natureza_gold IS 'Dimensão Gold: natureza do voo. 2 tipos: DOMÉSTICA e INTERNACIONAL. Fonte: dados_anac_silver.';
COMMENT ON COLUMN workspace.default.dim_natureza_gold.NATUREZA_ID IS 'Chave primária sequencial, ordenada por NATUREZA';
COMMENT ON COLUMN workspace.default.dim_natureza_gold.NATUREZA IS 'Tipo de natureza: DOMÉSTICA ou INTERNACIONAL';

COMMENT ON TABLE workspace.default.dim_grupo_voo_gold IS 'Dimensão Gold: grupo do voo. 4 tipos. Fonte: dados_anac_silver.';
COMMENT ON COLUMN workspace.default.dim_grupo_voo_gold.GRUPO_VOO_ID IS 'Chave primária sequencial, ordenada por GRUPO_DE_VOO';
COMMENT ON COLUMN workspace.default.dim_grupo_voo_gold.GRUPO_DE_VOO IS 'Tipo de grupo: IMPRODUTIVO, NÃO IDENTIFICADO, NÃO REGULAR ou REGULAR';

COMMENT ON TABLE workspace.default.fato_voos_gold IS 'Tabela Fato Gold: voos da ANAC. 1.096.174 registros com FKs para 4 dimensões. Fonte: dados_anac_silver.';
COMMENT ON COLUMN workspace.default.fato_voos_gold.VOO_ID IS 'Chave primária sequencial do voo';
COMMENT ON COLUMN workspace.default.fato_voos_gold.EMPRESA_ID IS 'FK para dim_empresa_gold';
COMMENT ON COLUMN workspace.default.fato_voos_gold.AEROPORTO_ORIGEM_ID IS 'FK para dim_aeroporto_gold. NULL em 5.212 voos internacionais';
COMMENT ON COLUMN workspace.default.fato_voos_gold.AEROPORTO_DESTINO_ID IS 'FK para dim_aeroporto_gold';
COMMENT ON COLUMN workspace.default.fato_voos_gold.NATUREZA_ID IS 'FK para dim_natureza_gold';
COMMENT ON COLUMN workspace.default.fato_voos_gold.GRUPO_VOO_ID IS 'FK para dim_grupo_voo_gold. NULL em 2 registros';
COMMENT ON COLUMN workspace.default.fato_voos_gold.ANO IS 'Ano do voo';
COMMENT ON COLUMN workspace.default.fato_voos_gold.MES IS 'Mês do voo';
COMMENT ON COLUMN workspace.default.fato_voos_gold.MES_ANO IS 'Combinação mês/ano no formato MM/AAAA';
COMMENT ON COLUMN workspace.default.fato_voos_gold.PASSAGEIROS_PAGOS IS 'Total de passageiros pagos. NULL em 3,7% dos registros';
COMMENT ON COLUMN workspace.default.fato_voos_gold.PASSAGEIROS_GRATIS IS 'Total de passageiros grátis. NULL em 3,7% dos registros';
COMMENT ON COLUMN workspace.default.fato_voos_gold.DISTANCIA_VOADA_KM IS 'Distância voada em km. NULL em 21,8% dos registros';
COMMENT ON COLUMN workspace.default.fato_voos_gold.ASSENTOS IS 'Total de assentos. NULL em 21,7% dos registros';
COMMENT ON COLUMN workspace.default.fato_voos_gold.DECOLAGENS IS 'Quantidade de decolagens realizadas na etapa de voo';

In [0]:
CREATE OR REPLACE VIEW workspace.default.vw_voos_completos AS
SELECT
    f.VOO_ID,
    f.ANO,
    f.MES,
    f.MES_ANO,

    e.EMPRESA_NOME,
    e.EMPRESA_NACIONALIDADE,

    ao.AEROPORTO_NOME AS AEROPORTO_ORIGEM_NOME,
    ao.PAIS AS AEROPORTO_ORIGEM_PAIS,

    ad.AEROPORTO_NOME AS AEROPORTO_DESTINO_NOME,
    ad.PAIS AS AEROPORTO_DESTINO_PAIS,

    n.NATUREZA,
    g.GRUPO_DE_VOO,

    f.PASSAGEIROS_PAGOS,
    f.PASSAGEIROS_GRATIS,
    f.DISTANCIA_VOADA_KM,
    f.ASSENTOS,
    f.DECOLAGENS

FROM workspace.default.fato_voos_gold f

LEFT JOIN workspace.default.dim_empresa_gold e
    ON f.EMPRESA_ID = e.EMPRESA_ID

LEFT JOIN workspace.default.dim_aeroporto_gold ao
    ON f.AEROPORTO_ORIGEM_ID = ao.AEROPORTO_ID

LEFT JOIN workspace.default.dim_aeroporto_gold ad
    ON f.AEROPORTO_DESTINO_ID = ad.AEROPORTO_ID

LEFT JOIN workspace.default.dim_natureza_gold n
    ON f.NATUREZA_ID = n.NATUREZA_ID

LEFT JOIN workspace.default.dim_grupo_voo_gold g
    ON f.GRUPO_VOO_ID = g.GRUPO_VOO_ID;